In [2]:
!pip install kaggle

Importing the dependencies

In [3]:
import os
import json

from zipfile import ZipFile
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

Data Collection

In [4]:
df = pd.read_csv(
    "/content/IMDB Dataset.csv",
    engine='python',
    on_bad_lines='skip'
)

In [5]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [7]:
df.replace({"sentiment":{"positive":1,"negative":0}},inplace=True)

/tmp/ipykernel_1353/1518912133.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({"sentiment":{"positive":1,"negative":0}},inplace=True)


In [8]:
df.sample(5)

,review,sentiment
40758,"A well-made and imaginative production, refres...",1
46864,FC De Kampioenen's only reason for existence i...,0
45393,Beware My Lovely originated from a play writte...,1
9872,Glacier Fox is one of the most heartrending an...,1
6942,This movie was probably about as silly as The ...,0


In [9]:
df["sentiment"].value_counts()

,count
sentiment,
1,25000
0,25000


In [10]:
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42)

In [11]:
train_df.shape

(40000, 2)

In [12]:
test_df.shape

(10000, 2)

In [15]:
tokenizer=Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_df['review'])
X_train=pad_sequences(tokenizer.texts_to_sequences(train_df['review']),maxlen=200)
X_test=pad_sequences(tokenizer.texts_to_sequences(test_df['review']),maxlen=200)

In [16]:
print(X_train)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [17]:
X_test

array([[   0,    0,    0, ...,  995,  719,  155],
       [  12,  162,   59, ...,  380,    7,    7],
       [   0,    0,    0, ...,   50, 1088,   96],
       ...,
       [   0,    0,    0, ...,  125,  200, 3241],
       [   0,    0,    0, ..., 1066,    1, 2305],
       [   0,    0,    0, ...,    1,  332,   27]], dtype=int32)

In [18]:
Y_train=train_df['sentiment']
Y_test=test_df['sentiment']

In [19]:
Y_train

,sentiment
39087,0
30893,0
45278,1
16398,0
13653,0
...,...
11284,1
44732,1
38158,0
860,1


In [20]:
Y_test

,sentiment
33553,1
9427,1
199,0
12447,1
39489,0
...,...
28567,0
25079,1
18707,1
15200,0


In [22]:


model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation="sigmoid"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [23]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
# compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [25]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 228s 441ms/step - accuracy: 0.7790 - loss: 0.4653 - val_accuracy: 0.8357 - val_loss: 0.3761
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 262s 450ms/step - accuracy: 0.8190 - loss: 0.4132 - val_accuracy: 0.8482 - val_loss: 0.3609
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 238s 476ms/step - accuracy: 0.8017 - loss: 0.4325 - val_accuracy: 0.8384 - val_loss: 0.3978
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 247s 446ms/step - accuracy: 0.8600 - loss: 0.3425 - val_accuracy: 0.8574 - val_loss: 0.3451
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 221s 443ms/step - accuracy: 0.8673 - loss: 0.3222 - val_accuracy: 0.8503 - val_loss: 0.3605


In [26]:
loss, accuracy = model.evaluate(X_test, Y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 130ms/step - accuracy: 0.8506 - loss: 0.3590
Test Loss: 0.35896408557891846
Test Accuracy: 0.850600004196167


In [27]:
def predict(review):
  sequence=tokenizer.texts_to_sequences([review])
  padded_sequence=pad_sequences(sequence,maxlen=200)
  prediction=model.predict(padded_sequence)
  if prediction[0][0] > 0.5:
    return "Positive"
  else:
    return "Negative"

In [28]:
rev="The movie was really fantastic, I enjoyed alot really alot, it was feeling like a rollerccoaster ride"
predict(rev)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 648ms/step


'Positive'

In [32]:
rev2="The movie so fcking trash that I hated every bit of it."
predict(rev2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step


'Negative'